# Clase 1 — Adquisición, histogramas y mejoramiento

**Unidad VII · 4 h · grupos de 3**

> **La pregunta de hoy:** ¿de dónde sale una imagen, cómo sé si sirve, y qué puedo hacer si no sirve?

## 1. Objetivos

Al terminar este cuaderno tienes que poder:

1. Obtener una imagen desde cuatro procedencias y explicar en qué se diferencian.
2. Describir una imagen como matriz NumPy: forma, ejes, canales, tipo.
3. Leer un histograma **y decir qué no puede medir**.
4. Convertir un histograma en un criterio de aceptación automático.
5. Aplicar cuatro realces y **medir** si sirvieron para algo.

Este cuaderno se ejecuta de arriba abajo. Si algo falla, para y arréglalo antes de seguir.

## Preparación

Funciona en dos sitios:

* **En tu máquina**, con el repositorio del motor clonado: usa imágenes reales del videojuego.
* **En Google Colab**, sin nada: degrada a datos sintéticos y el cuaderno sigue siendo ejecutable de principio a fin.

No hay celdas con `!pip install` ni `!git clone` a propósito: se comprueba y se avisa, en Python, para que la misma celda valga en los dos sitios.

In [ ]:
import os
import sys
from pathlib import Path

# Sin pantalla: el motor arranca en modo headless. Hay que fijarlo ANTES de
# importar pygame, porque SDL elige el controlador de vídeo al inicializarse.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")


def preparar_rutas() -> bool:
    """Pone `cvcourse` en el path. Devuelve si además hay motor."""
    for candidato in (Path.cwd(), *Path.cwd().parents):
        if (candidato / "cvcourse").is_dir():
            for ruta in (candidato, candidato.parent):
                if str(ruta) not in sys.path:
                    sys.path.insert(0, str(ruta))
            return True
    return False


if not preparar_rutas():
    print("No encuentro `cvcourse`. En Colab, clona el repositorio primero:")
    print("    !git clone <url-del-repositorio>")
    print("    %cd legacyofInfest/computer-vision-course")

from cvcourse import acquisition, engine_bridge, synthetic, viz

HAY_MOTOR = engine_bridge.hay_motor()
print(f"cvcourse listo. Motor disponible: {HAY_MOTOR}")
if not HAY_MOTOR:
    print("Sin motor: los apartados del videojuego usarán piezas sintéticas.")

## 2. El concepto: una imagen es una matriz

Todo lo demás del curso son operaciones sobre números. Una imagen en color es un array de tres dimensiones:

```
imagen[fila, columna, canal]
         ↓       ↓       ↓
       alto    ancho    R,G,B
```

**Dos trampas que vas a encontrar hoy mismo:**

1. **OpenCV entrega BGR**, no RGB. Si mezclas los convenios, el rojo sale azul. No falla: miente.
2. **`pygame.surfarray` entrega (ancho, alto, 3)**, al revés que todo lo demás. Con un sprite cuadrado ni lo notas — hasta que Sobel encuentra los bordes verticales donde estaban los horizontales.

`cvcourse` hace las dos conversiones una sola vez, y por eso todo lo que sale de aquí es **RGB, (alto, ancho, 3), uint8**.

In [ ]:
import numpy as np

imagen, verdad = synthetic.pieza_individual(tamano=128, clase="OK", semilla=0)

print(f"forma  : {imagen.shape}")
print(f"tipo   : {imagen.dtype}")
print(f"mínimo : {imagen.min()}   máximo: {imagen.max()}")
print(f"\nLa esquina superior izquierda, 4x4 píxeles:\n{imagen[:4, :4]}")

# Los píxeles son números y se pueden tocar como tales.
copia = imagen.copy()
copia[10:30, 10:30] = 255
viz.comparar(imagen, copia, "original", "con un cuadrado escrito a mano");

## 3. Fundamento: qué es exactamente un histograma

El histograma de una imagen $I$ con niveles $0..L-1$ es el vector $h$ donde

$$h[k] = \#\{(i,j) : I(i,j) = k\}$$

es decir, **cuántos píxeles valen exactamente $k$**. Normalizado por el número de píxeles, $p[k] = h[k]/N$, es la probabilidad de que un píxel tomado al azar valga $k$.

De esa definición salen las tres cosas que hay que entender hoy:

| Propiedad | Consecuencia |
|---|---|
| Suma $\sum_k h[k] = N$ | El histograma no pierde píxeles, sólo los reordena por valor |
| **Descarta $(i,j)$** | No sabe *dónde* está cada píxel. Por eso no puede ver ruido, enfoque ni forma |
| Depende sólo de la distribución | Dos imágenes completamente distintas pueden tener el mismo histograma |

La segunda fila es la más importante de la clase, y la vas a comprobar tú en el apartado 7.

In [ ]:
# Comprobemos las tres propiedades en lugar de creérnoslas.
gris = imagen.astype(np.uint8)
h = np.bincount(gris.ravel(), minlength=256)

print(f"suma del histograma : {h.sum()}")
print(f"píxeles de la imagen: {gris.size}")
print(f"¿coinciden?          {h.sum() == gris.size}")

# Segunda propiedad: barajar los píxeles destruye la imagen y NO cambia el histograma.
rng = np.random.default_rng(0)
barajada = gris.ravel().copy()
rng.shuffle(barajada)
barajada = barajada.reshape(gris.shape)

print(f"\n¿mismo histograma tras barajar? {np.array_equal(np.bincount(barajada.ravel(), minlength=256), h)}")
viz.comparar(gris, barajada, "original", "píxeles barajados — MISMO histograma");

**Párate aquí.** Esas dos imágenes tienen histogramas idénticos y no se parecen en nada. Cualquier cosa que decidas mirando sólo el histograma, la estás decidiendo sin mirar la imagen.

## 4. Ejemplo básico: adquirir desde varias procedencias

El mismo código de proceso, cuatro orígenes. Si tu función de análisis necesita saber de dónde vino la imagen, está mal escrita.

In [ ]:
def describir(imagen, nombre):
    """Las cinco cifras con las que se juzga una toma."""
    g = imagen.mean(axis=2) if imagen.ndim == 3 else imagen
    return {
        "fuente": nombre,
        "forma": imagen.shape,
        "media": round(float(g.mean()), 1),
        "rango": round(float(g.max() - g.min())),
        "sat%": round(float((g >= 254).mean() * 100), 2),
        "neg%": round(float((g <= 1).mean() * 100), 2),
    }


candidatas = [("sintética", lambda: acquisition.FuenteSintetica(n=1, tamano=128))]
if HAY_MOTOR:
    candidatas += [
        ("motor: sprite", lambda: acquisition.FuenteMotor("sprites", maximo=1)),
        ("motor: escena", lambda: acquisition.FuenteMotor(escena="filter", maximo=1)),
    ]

imagenes, titulos = [], []
for nombre, constructor in candidatas:
    try:
        with constructor() as fuente:
            img = fuente.leer()
    except (OSError, RuntimeError) as error:
        print(f"[no disponible] {nombre}: {error}")
        continue
    if img is None:
        continue
    print(describir(img, nombre))
    imagenes.append(img)
    titulos.append(f"{nombre}\n{img.shape[1]}x{img.shape[0]}")

viz.rejilla(imagenes, titulos, columnas=3, titulo_general="Cuatro procedencias, un solo pipeline");

## 5. Aplicación industrial: el histograma como decisión, no como gráfico

En un laboratorio el histograma se mira. En una línea de producción **no lo mira nadie**: la cámara toma una imagen cada 200 ms y hay que decidir sola.

Rechazar una imagen mala cuesta 200 ms. Procesarla y dar una coordenada equivocada cuesta una pieza.

In [ ]:
def aceptar(imagen, max_sat=2.0, max_neg=5.0, min_rango=60.0, media_valida=(40, 215)):
    """Devuelve (aceptada, motivos). Los motivos importan: 'rechazada' a secas
    no le dice al operario si bajar la exposición o revisar la iluminación."""
    g = imagen.mean(axis=2) if imagen.ndim == 3 else imagen.astype(float)
    motivos = []
    if (g >= 254).mean() * 100 > max_sat:
        motivos.append(f"saturación {(g >= 254).mean() * 100:.1f}%")
    if (g <= 1).mean() * 100 > max_neg:
        motivos.append(f"subexposición {(g <= 1).mean() * 100:.1f}%")
    if g.max() - g.min() < min_rango:
        motivos.append(f"rango {g.max() - g.min():.0f}")
    if not media_valida[0] <= g.mean() <= media_valida[1]:
        motivos.append(f"media {g.mean():.0f}")
    return (not motivos), motivos


base, _ = synthetic.pieza_individual(tamano=192, semilla=1)
casos = {
    "correcta": base,
    "sobreexpuesta": np.clip(base * 2.4, 0, 255).astype(np.uint8),
    "subexpuesta": np.clip(base * 0.25, 0, 255).astype(np.uint8),
    "sin contraste": np.clip(base * 0.25 + 100, 0, 255).astype(np.uint8),
}

for nombre, img in casos.items():
    ok, motivos = aceptar(img)
    print(f"{nombre:16s} {'ACEPTA' if ok else 'RECHAZA'}  {', '.join(motivos)}")

viz.rejilla(list(casos.values()), list(casos), columnas=4, titulo_general="Aceptación de toma");

## 6. Aplicación al videojuego: el histograma de un sprite miente

Los sprites del motor son PNG **RGBA**: el cuarto canal dice qué píxeles son transparentes. Casi todo el software de visión lo descarta, y al descartarlo **lo transparente se vuelve negro**.

Un sprite puede ser 75 % transparente. Ese 75 % aparece como un pico gigantesco en el valor 0, y a partir de ahí la media, el contraste y el umbral que elijas están mal. Sin ninguna excepción, sin ningún aviso.

In [ ]:
if HAY_MOTOR:
    from PIL import Image

    ruta = engine_bridge.raiz_del_repositorio() / "assets/sprites/player/player_idle.png"
    rgba = np.asarray(Image.open(ruta).convert("RGBA"))
    gris_sprite = rgba[:, :, :3].mean(axis=2)
    opaco = rgba[:, :, 3] > 0

    ingenua, correcta = gris_sprite.mean(), gris_sprite[opaco].mean()
    print(f"media ingenua  : {ingenua:6.1f}  (sobre {gris_sprite.size} píxeles del lienzo)")
    print(f"media correcta : {correcta:6.1f}  (sobre {int(opaco.sum())} con dibujo)")
    print(f"error          : {(correcta - ingenua) / correcta * 100:5.0f} %")

    viz.rejilla(
        [rgba[:, :, :3], (opaco * 255).astype(np.uint8)],
        ["RGB: lo transparente ya es negro", "canal alfa: qué existe de verdad"],
        columnas=2,
    )
else:
    print("Sin motor. Este apartado necesita los sprites del videojuego.")
    print("La idea igual: si tu imagen tiene píxeles que 'no existen',")
    print("el histograma los cuenta como si valieran 0.")

## 7. Experimento: lo que el histograma no puede ver

Vas a construir una imagen **mala de forma evidente** que pase los cuatro criterios del apartado 5.

Sube `sigma` poco a poco y observa dos cosas a la vez: cuándo la imagen deja de servir para nada, y cuándo el criterio se entera. **No coinciden.**

In [ ]:
rng = np.random.default_rng(7)

print(f"{'sigma':>6s} {'sat%':>6s} {'neg%':>6s} {'rango':>6s} {'media':>6s}  veredicto")
print("-" * 56)
ruidosas, etiquetas = [], []
for sigma in (0, 5, 10, 17, 25, 40):
    sucia = np.clip(base.astype(float) + rng.normal(0, sigma, base.shape), 0, 255).astype(np.uint8)
    ok, motivos = aceptar(sucia)
    g = sucia.astype(float)
    print(
        f"{sigma:>6d} {(g >= 254).mean() * 100:>6.2f} {(g <= 1).mean() * 100:>6.2f} "
        f"{g.max() - g.min():>6.0f} {g.mean():>6.0f}  {'ACEPTA' if ok else 'RECHAZA'}"
    )
    ruidosas.append(sucia)
    etiquetas.append(f"sigma={sigma}\n{'ACEPTA' if ok else 'RECHAZA'}")

viz.rejilla(ruidosas, etiquetas, columnas=6, titulo_general="El ruido que el histograma no ve");

**¿Por qué?** Porque el ruido gaussiano es simétrico: le suma a unos píxeles lo que le resta a otros. La *distribución* apenas cambia, y el histograma **sólo ve la distribución**. Lo que cambia es *dónde* está cada valor, y eso el histograma lo tiró en su definición (apartado 3).

Lo que sí lo detecta es un filtro espacial, que mira vecindarios en lugar de valores sueltos. Eso es la Clase 2.

## 8. Ejercicio guiado: los cuatro realces

Completa las cuatro funciones. Las tres primeras son de una línea; la cuarta necesita la suma acumulada del histograma.

| Realce | Qué le hace al histograma |
|---|---|
| brillo | lo **desplaza** |
| contraste | lo **ensancha** alrededor de la media |
| estiramiento | lo lleva a ocupar todo `[0, 255]` |
| ecualización | lo **aplana** (no lineal) |

In [ ]:
def subir_brillo(imagen, delta=60.0):
    return np.clip(imagen.astype(float) + delta, 0, 255).astype(np.uint8)


def subir_contraste(imagen, factor=2.0):
    media = imagen.mean()
    return np.clip((imagen.astype(float) - media) * factor + media, 0, 255).astype(np.uint8)


def estirar(imagen):
    lo, hi = float(imagen.min()), float(imagen.max())
    if hi <= lo:
        return imagen.copy()
    return (((imagen.astype(float) - lo) / (hi - lo)) * 255).astype(np.uint8)


def ecualizar(imagen):
    h = np.bincount(imagen.ravel(), minlength=256)
    acumulado = h.cumsum().astype(float)
    acumulado = (acumulado - acumulado.min()) / max(acumulado.max() - acumulado.min(), 1)
    return (acumulado[imagen] * 255).astype(np.uint8)


subexpuesta = np.clip(base.astype(float) * 0.32 + 8, 0, 255).astype(np.uint8)
realces = {
    "sin realce": subexpuesta,
    "brillo +60": subir_brillo(subexpuesta),
    "contraste x2": subir_contraste(subexpuesta),
    "estirado": estirar(subexpuesta),
    "ecualizado": ecualizar(subexpuesta),
}
viz.rejilla(list(realces.values()), list(realces), columnas=5, titulo_general="Cuatro realces");

## 9. Ejercicio grupal: ¿mejoró de verdad?

Aquí está la idea que sostiene el resto del curso.

Mide cada realce con **dos varas distintas**:

1. el histograma — rango, desviación, cuántos niveles se ocupan;
2. **la tarea** — umbraliza con Otsu y compara contra la verdad-terreno con IoU.

Antes de ejecutar la celda, **apuesta en voz alta con tu grupo cuál va a ganar**. Anótalo.

In [ ]:
from skimage.filters import threshold_otsu

pieza, verdad_pieza = synthetic.pieza_individual(
    tamano=192, clase="NO_OK", defecto="mota", semilla=4
)
f0, c0, f1, c1 = verdad_pieza.bbox
verdad = np.zeros(pieza.shape, dtype=bool)
verdad[f0:f1, c0:c1] = True

mala = np.clip(pieza.astype(float) * 0.32 + 8, 0, 255).astype(np.uint8)


def iou_con_otsu(imagen, verdad):
    umbral = float(threshold_otsu(imagen))
    mascara = imagen > umbral
    union = float((mascara | verdad).sum())
    return umbral, (float((mascara & verdad).sum()) / union if union else 0.0)


print(f"{'realce':14s} {'rango':>6s} {'desv':>6s} {'ocup%':>6s} {'Otsu':>6s} {'IoU':>6s}")
print("-" * 52)
for nombre, fn in [
    ("sin realce", lambda x: x),
    ("brillo +60", subir_brillo),
    ("contraste x2", subir_contraste),
    ("estirado", estirar),
    ("ecualizado", ecualizar),
]:
    img = fn(mala)
    umbral, iou = iou_con_otsu(img, verdad)
    ocupacion = (np.bincount(img.ravel(), minlength=256) > 0).mean() * 100
    print(
        f"{nombre:14s} {img.max() - img.min():>6d} {img.std():>6.1f} "
        f"{ocupacion:>6.1f} {umbral:>6.0f} {iou:>6.3f}"
    )

### Lo que acabas de medir

El histograma mejora muchísimo. **El IoU no se mueve.** No es un fallo del ejercicio: es un teorema pequeño.

> Umbralizar es comparar cada píxel con un número. Los cuatro realces son funciones **monótonas crecientes**, y una función monótona no cambia el *orden* de dos píxeles. Así que la partición que produce el umbral es la misma: lo único que se mueve es en qué valor cae. Mira la columna Otsu.

La ecualización es la excepción, y va **a peor**. Es monótona pero no inyectiva: al redondear a 8 bits colapsa niveles vecinos, y ahí sí se destruye información.

**Entonces, ¿para qué sirve realzar?** Para que un humano lea la imagen, y para lo que venga después que no sea un umbral global — filtros, gradientes, umbral adaptativo (Clases 2 y 3). Lo que arregla una toma subexpuesta es la **exposición**, no el post-proceso: el realce estira lo que hay, no añade lo que falta.

## 10. Reto

Elige uno:

**A · Histograma consciente del alfa.** Escribe `histograma_con_alfa(ruta)` que devuelva el histograma correcto de un sprite RGBA y demuestre con una cifra cuánto se equivocaba el ingenuo.

**B · Rompe el criterio de otra manera.** El ruido no es el único caso que se le escapa a `aceptar()`. Encuentra otro: una imagen mala, por un motivo **distinto**, que pase los cuatro filtros. Explica qué propiedad no puede ver un histograma.

**C · Ecualización adaptativa.** Busca `skimage.exposure.equalize_adapthist` (CLAHE). Repite el apartado 9 con ella. ¿Cambia el IoU? Si cambia, ¿por qué escapa al teorema de arriba?

In [ ]:
# Tu reto aquí.


## 11. Preguntas de análisis

Respóndelas en `analisis.md`. Cada una con una cifra o una figura detrás.

1. Tus imágenes tienen formas distintas. ¿Cuál dimensión es el alto y cuál el ancho, y cómo lo comprobaste **sin** mirar la imagen?
2. ¿Por qué el histograma de un sprite con transparencia tiene un pico enorme en 0? ¿Es un defecto de la imagen, del cargador, o de la pregunta que le estamos haciendo?
3. Tu criterio del apartado 5 acepta una imagen mala. ¿Qué propiedad de esa imagen no puede ver un histograma, y por qué no puede verla? Cita la definición del apartado 3.
4. Tras el realce, ¿mejoró el histograma? ¿Mejoró la tarea? Si las respuestas no coinciden, explica el porqué.
5. ¿Cuál de tus imágenes **no** podrías volver a obtener idéntica mañana? ¿Qué consecuencia tiene eso para poder corregir este laboratorio?

## 12. Conclusiones

1. Una imagen es una matriz. El convenio del curso es **RGB, (alto, ancho, 3), uint8**, y las conversiones se hacen una sola vez.
2. El histograma cuenta **cuántos** píxeles hay de cada valor y descarta **dónde** están. Diagnostica exposición y contraste; no puede diagnosticar ruido, enfoque ni forma.
3. Un histograma con umbrales es un **criterio de aceptación automático**, y en producción eso vale más que un gráfico.
4. **Realzar no crea información.** Y umbralizar es invariante a transformaciones monótonas, así que el realce previo a un umbral global no cambia el resultado.
5. Lo que arregla una toma mala es la adquisición.

**Clase 2:** si el histograma no ve el ruido, ¿qué lo ve? Filtros espaciales, convolución, y de ahí a los bordes: Sobel y Canny.

## Bibliografía

- Gonzalez, R. C. y Woods, R. E. *Digital Image Processing*, 4.ª ed. Capítulo 3 (realce en el dominio espacial) y §3.3 (procesamiento de histograma).
- Szeliski, R. *Computer Vision: Algorithms and Applications*, 2.ª ed. §3.1 (operadores puntuales).
- Documentación de scikit-image: módulo `exposure`.
- Código del motor: `src/framework/processing/filter_tools.py` — la implementación que usan las escenas-laboratorio de la Unidad VII.